# nilearn-aal-viewer — Demo

This notebook demonstrates `show_brain_viewer` on a **synthetic t-map**  
(no real patient data required).

To use your own statistical map, replace `tmap_nii` with any NIfTI  
t-map or z-map in MNI space.

In [1]:
import sys
sys.path.insert(0, '..')   # allow import from repo root

import numpy as np
import nibabel as nib
from pathlib import Path
from nilearn import datasets

from viewer import build_aal_lookup, lut_to_json, show_brain_viewer

## 1 — Atlas path

Download AAL3v1 from:  
https://www.gin.cnrs.fr/en/tools/aal/

Then set the path below.

In [2]:
AAL_PATH = Path(r'C:\Users\flore\OneDrive\Desktop\DynaKPET\templates\atlas\AAL3v1_1mm.nii.gz')  # <-- set this
TMAP_PATH = Path('./tmap.nii.gz')  # <-- set this

## 2 — Build the lookup table (once per session)

In [3]:
lut      = build_aal_lookup(aal_path=AAL_PATH, step_mm=2)
lut_json = lut_to_json(lut)
print(f'{len(lut)} voxels indexed, {len(set(lut.values()))} regions')

  AAL3 labels loaded : 170 regions (AAL3v1_1mm.nii.txt)
  LUT built : 185713 voxels, 165 regions, resolution 2 mm
185713 voxels indexed, 165 regions


## 3 — Load your t-map

In [4]:
## 3 — Load your t-map
tmap_nii = nib.load(TMAP_PATH)
print('T-map shape:', tmap_nii.shape)

T-map shape: (79, 95, 79)


## 4 — Generate the viewer

In [5]:
OUT_DIR = Path('../outputs')
bg_img  = datasets.load_mni152_template()

t_thresh = 3.5
show_brain_viewer(
    img            = tmap_nii,
    threshold      = t_thresh,
    title          = 'Demo contrast',
    contrast_label = 'Example data',
    fname          = 'viewer_demo.html',
    out_dir        = OUT_DIR,
    lut_json       = lut_json,
    step           = 2,
    bg_img         = bg_img,
    cmap           = 'cold_hot',
    opacity        = 0.8,
    height         = 520,
)

C:\Users\flore\anaconda3\envs\petenv2\Lib\site-packages\numpy\core\fromnumeric.py:771: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


  Saved : viewer_demo_aal.html


'..\\outputs\\viewer_demo_aal.html'

### 4bis — And if you also want the descriptive table for the clusters

In [6]:
from nilearn.reporting import get_clusters_table

t_thresh = 3.5
tbl = get_clusters_table(tmap_nii, stat_threshold=t_thresh, cluster_threshold=50)

show_brain_viewer(
    img                  = tmap_nii,
    threshold            = t_thresh,
    title                = "Demo contrast",
    contrast_label       = "p<0.001 uncorrected",
    fname                = "viewer_CB.html",
    out_dir              = OUT_DIR,
    lut_json             = lut_json,
    lut                  = lut,          # ← active colonne ROI + clic→navigation
    step                 = 2,
    bg_img               = bg_img,
    clusters_table       = tbl,
    clusters_table_title = "Clusters in AAL3 atlas — p<0.001",
)

C:\Users\flore\anaconda3\envs\petenv2\Lib\site-packages\numpy\core\fromnumeric.py:771: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


  Saved : viewer_CB_aal.html


'..\\outputs\\viewer_CB_aal.html'